# LSTM Reimplementation

This notebook reproduces the LSTM-based fake news detection model from:

> Liu, S. (2023). *Social Media Fake Information Identification Method Based on LSTM*. GEFHR 2023, Vol. 21, pp. 703–709.

All model parameters follow the original paper exactly (Table 2). The goal is to verify that the reported 99% accuracy and 100% AUC are reproducible, and to understand what drives those numbers given the dataset bias identified in `00_eda.ipynb`.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

# Keras is now integrated into TensorFlow (tf.keras = Keras)
# keras.preprocessing.text was removed in Keras 3 — use tensorflow.keras instead
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

os.makedirs('../results', exist_ok=True)
print('TensorFlow:', tf.__version__)
print('Keras:', tf.keras.__version__)

## 2. Load Data

In [ ]:
fake = pd.read_csv('../data/raw/Fake.csv')
true = pd.read_csv('../data/raw/True.csv')

fake['label'] = 0
true['label'] = 1

df = pd.concat([fake, true]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Total: {len(df):,} rows')
print(df['label'].value_counts())

原论文使用 `title + text` 作为输入文本，`subject` 和 `date` 列并未显式排除，因此模型可能接触到了泄露特征（见 `00_eda.ipynb`）。为忠实复现原论文，这里沿用相同设置——使用 `text` 列作为输入。

The original paper uses the article text as input without explicitly removing `subject` or `date`. To faithfully reproduce the paper's results, we follow the same setup here and use the `text` column as input.

## 3. Text Preprocessing

按照原论文的流程，文本预处理分三步：划分训练测试集 → Tokenizer 建立词表 → 将文本转换为数字序列并统一长度（padding）。

Following the original paper, preprocessing has three steps: train/test split → Tokenizer vocabulary construction → convert text to padded integer sequences.

In [ ]:
# Train/test split — 80/20, matching original paper
X = df['text'].fillna('')
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

In [ ]:
# Tokenizer — num_words=5000 matches original paper
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(X_train)

vocab_size = len(tokenizer.word_index)
print(f'Full vocabulary size: {vocab_size:,}')
print(f'Using top 5,000 words (as per original paper)')

In [ ]:
# Convert text to sequences and pad to length 200 — matches original paper
X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=200)
X_test_seq  = pad_sequences(tokenizer.texts_to_sequences(X_test),  maxlen=200)

print(f'X_train shape: {X_train_seq.shape}')
print(f'X_test shape:  {X_test_seq.shape}')

预处理完成后，每篇文章被表示为一个长度为 200 的整数数组。数组里每个数字是词在词表中的编号，不足 200 词的文章用 0 补齐（zero-padding），超过 200 词的文章被截断。这个格式就是 LSTM 模型接受的输入形状：`(样本数, 200)`。

After preprocessing, each article is represented as an integer array of length 200. Each integer is the index of a word in the vocabulary. Articles shorter than 200 words are zero-padded; longer ones are truncated. The resulting shape `(n_samples, 200)` is the direct input to the LSTM model.